# Test Problem 1: FD with weights generated by a neural network

Weights of one stencil: $\hat w = w_0 + (I-\Pi)v$, $w_0 = P(P^TP)^{-1}Lp$, $v$ = network output (exact on the polynomials of degree `deg`).

Loss: $L = \|A^{-1}b - U^*\|^2/\|U^*\|^2 + \lambda\,\mathrm{mean}_i\sum_j |\hat w_{ij}|\,\|\hat y_j\|^{\mu}$ (second term: eq. (20) of Davydov & Schaback).

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from learned_fd_TP1 import make_nets, train_TP1, learned_fd_TP1

In [ ]:
deg     = 4            # polynomial degree
ntype   = 'uniform'    # node type: 'uniform' or 'halton'
Q       = (deg + 1) * (deg + 2) // 2
nloc    = 3 * Q        # stencil size
m_train = 2            # node level used for training
steps   = 500          # training steps (Adam)
lr      = 1e-3         # learning rate
lam     = 0.1          # weight of the eq. (20) term
mu      = deg          # exponent in eq. (20)
mlist   = range(0, 5)  # node levels X^(m)
msigma  = 3            # compute sigma only up to this level

In [ ]:
torch.manual_seed(0)
nets = make_nets(deg, nloc)
hist = train_TP1(nets, m_train, deg, nloc, ntype, lam=lam, mu=mu, steps=steps, lr=lr, log=50)

In [ ]:
H = np.array(hist)
plt.semilogy(H[:, 0], label='loss'); plt.semilogy(H[:, 1], label='error term'); plt.semilogy(lam * H[:, 2], label='lam * eq.(20) term')
plt.xlabel('step'); plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
print(f'NN weights, degree {deg} (Q = {Q}), nloc = {nloc}, {ntype} nodes')
print(f"{'m':>3} {'h':>8} {'Nint':>7} {'rms':>10} {'max':>10} {'sigma':>10} {'nnz/row':>8} {'time[s]':>8}")
res = []
for m in mlist:
    o = learned_fd_TP1(m, deg, nets, nloc, ntype, do_sigma=m <= msigma, mu=mu)
    res.append(o)
    print(f"{m:3d} {o['h']:8.5f} {o['Nint']:7d} {o['rms']:10.3e} {o['max']:10.3e} {o['sigma']:10.2e} {o['nnz_row']:8.1f} {o['time']:8.1f}")

In [ ]:
N = [o['Nint'] for o in res]
plt.loglog(N, [o['rms'] for o in res], 'o-')
plt.xlabel('$N_{int}$'); plt.ylabel('rms error'); plt.grid(alpha=0.3)
plt.title(f'Test Problem 1, NN weights, degree {deg}, {ntype} nodes'); plt.show()